<a href="https://colab.research.google.com/github/CenkAydin/TDL-ADD/blob/main/TDL_Mamba_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TDL-ADD — Aşama 2: Mamba Eğitimi (Colab)
**Hücre çalıştırma sırası:** 1 → 2 → 3 → 4  
Hücre 3 uzun sürer (~30–60 dk preprocess). Runtime yeniden başlatılırsa sadece Hücre 4'ü çalıştırabilirsiniz (özellikler `/content/` diskinde duruyorsa).

In [1]:
# ── Hücre 1: Drive Bağlama + Repo Klonlama ────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Checkpoint ve score çıktıları için Drive klasörlerini önceden oluştur
import os
os.makedirs('/content/drive/MyDrive/TDL-ADD/models/A2_Mamba/checkpoints_A2_Mamba', exist_ok=True)
os.makedirs('/content/drive/MyDrive/TDL-ADD/scores/A2_Mamba', exist_ok=True)

# Repo klonla (zaten varsa atla)
if not os.path.exists('/content/TDL-ADD'):
    !git clone https://github.com/CenkAydin/TDL-ADD /content/TDL-ADD
else:
    print('Repo zaten mevcut, güncelleniyor...')
    !git -C /content/TDL-ADD pull

%cd /content/TDL-ADD
print('Çalışma dizini:', os.getcwd())

Mounted at /content/drive
Cloning into '/content/TDL-ADD'...
remote: Enumerating objects: 200, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 200 (delta 4), reused 15 (delta 3), pack-reused 179 (from 1)
Receiving objects: 100% (200/200), 1.06 GiB | 32.58 MiB/s, done.
Resolving deltas: 100% (67/67), done.
/content/TDL-ADD
Çalışma dizini: /content/TDL-ADD


In [2]:
import os
os.environ['CUDA_HOME']                 = '/usr/local/cuda'
os.environ['CAUSAL_CONV1D_FORCE_BUILD'] = 'TRUE'
os.environ['MAMBA_FORCE_BUILD']         = 'TRUE'
os.environ['MAX_JOBS']                  = '4'

!nvcc --version
!python -c "import torch; print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)"

!pip install packaging ninja
!pip install causal-conv1d --no-build-isolation
!pip install mamba-ssm --no-build-isolation

!python -c "from mamba_ssm import Mamba; print('mamba-ssm OK')"
!pip install transformers tqdm pytorch-model-summary --quiet

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
PyTorch: 2.9.0+cu126 | CUDA: 12.6
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 11.8 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
  Created wheel for causal-conv1d: filename=causal_conv1d-1.6.1-cp312-cp312-linux_x86_64.whl size=193269715 sha256=f26ade3b8fc16c1cdc6e8e1c46ad3b77e1c25a11562d78f8c3424d084ce39a3f
  Stored in directory: /root/.cache/pip/wheels/98/4a/75/b24971cff4599825b16b612f08fbd2e60a2c336a56e081a3c8
Successfully built causal-conv1d
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.7/121.7 kB 5.4 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
  Created wheel for mamba-ssm: filename=mamba_ssm-2.3.1-cp312-cp312-linux_x86_64.whl size=370556949 sha256=9505b35ce2f10746f6f59e452e6e13070a8ba8754c8b4edfb33e5f0e17092306
  Stored i

In [3]:
# ── Hücre 3: Veri İndirme + Açma + Preprocess ─────────────────────────────────
# Bu hücre ~30–60 dk sürebilir. Zenodo arşivi ~25 GB'dır.
import os

DATA_ROOT    = '/content/asv2019PS/database'
FEATURE_ROOT = '/content/asv2019PS/preprocess_A1_WavLM_Large'
ARCHIVE_PATH = '/content/asv2019ps_archive.zip'

os.makedirs(DATA_ROOT, exist_ok=True)

# 1) Zenodo arşivini indir (zaten varsa atla)
if not os.path.exists(ARCHIVE_PATH):
    print('Zenodo arşivi indiriliyor (~25 GB)...')
    !wget -q --show-progress \
        'https://zenodo.org/api/records/5766198/files-archive' \
        -O {ARCHIVE_PATH}
else:
    print('Arşiv zaten mevcut, indirme atlandı.')

# 2) Arşivi aç
if not os.path.exists(os.path.join(DATA_ROOT, 'train')):
    print('Arşiv açılıyor...')
    !unzip -q {ARCHIVE_PATH} -d /content/asv2019PS_raw
    # Açılan klasör yapısını incele ve doğru konuma taşı
    !ls /content/asv2019PS_raw/
    # NOT: Arşiv içindeki klasör adına göre aşağıdaki komutu güncelle:
    # !mv /content/asv2019PS_raw/<KLASOR_ADI>/* {DATA_ROOT}/
else:
    print('Veri zaten açılmış.')

# 3) WavLM-Large ile özellik çıkartma
if not os.path.exists(os.path.join(FEATURE_ROOT, 'train', 'wavlm-large')):
    print('Özellik çıkartma başlıyor...')
    !python /content/TDL-ADD/preprocess.py \
        --database_dir {DATA_ROOT} \
        --protocol_dir /content/TDL-ADD/label \
        --output_dir   {FEATURE_ROOT}
else:
    print('Özellikler zaten çıkartılmış.')

Zenodo arşivi indiriliyor (~25 GB)...
/content/asv2019ps_     [                <=> ]   9.27G  49.8MB/s    in 2m 31s  
Arşiv açılıyor...
database_dev.tar.gz	   database_segment_labels_v1.2.tar.gz	README_v1.2
database_eval.tar.gz	   database_train.tar.gz
database_protocols.tar.gz  database_vad.tar.gz
Özellik çıkartma başlıyor...
/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1275: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:434.)
  _C._set_default_tensor_type(t)
2026-04-20 00:32:17.694849: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776645138.342629   46194 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for 

In [4]:
# ── Hücre 4: Eğitimi Başlat ───────────────────────────────────────────────────
# Çıktılar: checkpointler ve loglar doğrudan Drive'a yazılır.
!python /content/TDL-ADD/main_train.py \
    -m  TDL_Mamba \
    -f  /content/asv2019PS/preprocess_A1_WavLM_Large \
    -d  /content/asv2019PS/database \
    -o  /content/drive/MyDrive/TDL-ADD/models/A2_Mamba \
    --ckpt_subdir checkpoints_A2_Mamba \
    --num_epochs  200 \
    --batch_size  24 \
    --lr          0.0001 \
    --lam         0.1 \
    --num_workers 2 \
    --base_loss   bce \
    --gpu         0

/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1275: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:434.)
  _C._set_default_tensor_type(t)
Cuda device available:  True
  0% 0/200 [00:00<?, ?it/s]
Epoch: 1 

  0% 0/1058 [00:00<?, ?it/s]
  0% 0/200 [00:00<?, ?it/s]
Traceback (most recent call last):
  File "/content/TDL-ADD/main_train.py", line 275, in <module>
    _, _ = train(args)
           ^^^^^^^^^^^
  File "/content/TDL-ADD/main_train.py", line 174, in train
    featOri, audio_fnOri, lenOri, labelsOri = next(trainOri_flow)
                                              ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 732, in __next__
    data = self._next_data()
           ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-pac